In [2]:
!which python
!pip freeze > requirements.txt

/home/sa1/Documents/gitReps/pythonQuickAndDirty/venv/bin/python


In [13]:
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src" / "data" / "SmartEM" / "Philips").exists():
    repo_root = repo_root.parents[2]

data_dir = repo_root / "src" / "data" / "SmartEM" / "Philips"

input_data = pd.read_csv(data_dir / "input.csv")
output_data = pd.read_csv(data_dir / "output.csv")

# input_data, output_data

In [30]:
# input_data.nunique()
# Drop columns with only one unique value
input_data = input_data.loc[:, input_data.nunique() > 1]
input_data.nunique()
input_data.head()
# Number of rows in input_data and output_data
# len(input_data), len(output_data)

,input_1,input_2,input_3,input_4,input_5,input_6,input_7,input_8,input_9,input_10,input_11,input_14,input_18,input_19,input_20,input_21,input_22
0,C01,79,83,57,82,189,233,290-360-260,366-388-364,V-type,15,62.311333,-12.950,-1.136,930.932365,326.904678,21.103807
1,C01,79,83,57,82,189,233,290-360-260,366-388-364,V-type,15,55.742061,-12.239,-4.382,1082.927137,153.042818,13.931269
2,C01,79,83,57,82,189,233,290-360-260,366-388-364,V-type,15,67.942716,-10.024,8.277,984.849277,38.213696,18.031709
3,C01,79,83,57,82,189,233,290-360-260,366-388-364,V-type,15,45.419135,-11.168,6.654,983.234347,357.832140,11.502451
4,C01,79,83,57,82,189,233,290-360-260,366-388-364,V-type,15,61.599897,-12.991,0.487,704.864190,204.853589,20.680040


In [ ]:
from sklearn.model_selection import train_test_split # Function to split data into training and testing sets
import numpy as np

def trainTestSplit(input_data, output_data, number_of_trainingsamples, seed=None):
    """
    Split data based on unique values in input_1.

    Parameters
    ----------
    input_data : pandas.DataFrame
    output_data : pandas.DataFrame or pandas.Series
    number_of_trainingsamples : int
        Number of unique input_1 values to place in training set.
    seed : int or None
        Random seed used to shuffle the unique input_1 values.

    Returns
    -------
    train_inputs, test_inputs, train_outputs, test_outputs
    """

    # Find all unique configurations
    unique_configs = input_data["input_1"].unique()

    total_configs = len(unique_configs)

    if number_of_trainingsamples > total_configs:
        raise ValueError(
            f"number_of_trainingsamples={number_of_trainingsamples} is greater than "
            f"the number of unique input_1 values ({total_configs})"
        )
    # Shuffle the unique configurations
    rng = np.random.default_rng(seed)
    shuffled_configs = rng.permutation(unique_configs)

    # Select train and test configuration values
    train_configs = shuffled_configs[:number_of_trainingsamples]
    test_configs = shuffled_configs[number_of_trainingsamples:]

    # Create masks
    train_mask = input_data["input_1"].isin(train_configs)
    test_mask = input_data["input_1"].isin(test_configs)

    # Split input and output data
    train_inputs = input_data[train_mask]
    test_inputs = input_data[test_mask]

    train_outputs = output_data[train_mask]
    test_outputs = output_data[test_mask]

    return train_inputs, test_inputs, train_outputs, test_outputs


def categoricalPreProcessing(input_data):
    """
    Dummy preprocessing for categorical columns.
    Replaces columns input_2 to input_11 with -1.
    Parameters
    ----------
    input_data : pandas.DataFrame
    Returns
    -------
    pandas.DataFrame
    """
    processed_data = input_data.copy()
    categorical_columns = [f"input_{i}" for i in range(2, 12)]
    for col in categorical_columns:
        if col in processed_data.columns:
            processed_data[col] = -1

    return processed_data

def anglePreProcessing(input_data):
    """
    Convert angle columns from degrees to radians.
    input_14 and input_21 are assumed to be angles in degrees.
    """
    processed_data = input_data.copy()

    angle_columns = ["input_14", "input_21"]

    for col in angle_columns:
        if col in processed_data.columns:
            processed_data[col] = np.deg2rad(processed_data[col])

    return processed_data




from xgboost import XGBRegressor

def train_xgboost(X_train, y_train):
    model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    enable_categorical=True,
    verbosity=2
)

    model.fit(X_train, y_train, verbose=True)
    return model

from lightgbm import LGBMRegressor
def train_lightgbm(X_train, y_train):
    model = LGBMRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)
    return model

from catboost import CatBoostRegressor
def train_catboost(X_train, y_train, categorical_columns=None):
    model = CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=10
    )

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_columns
    )

    return model

In [29]:

# Xtrain, Xtest, Ytrain, Ytest = trainTestSplit(input_data, output_data)
train_inputs, test_inputs, train_outputs, test_outputs = trainTestSplit(
    input_data,
    output_data,
    number_of_trainingsamples=10,
    seed=42
)
# train_inputs, test_inputs = categoricalPreProcessing(train_inputs), categoricalPreProcessing(test_inputs)
# train_inputs, test_inputs = anglePreProcessing(train_inputs), anglePreProcessing(test_inputs)
train_inputs.nunique()
# train_inputs.describe()
# train_inputs.head()

input_1        10
input_2         5
input_3         3
input_4         7
input_5         3
input_6         7
input_7         3
input_8         6
input_9         4
input_10        2
input_11        2
input_14     1000
input_18       31
input_19       62
input_20     5741
input_21    22288
input_22    22286
dtype: int64

In [ ]:
X_train = Xtrain[0]
y_train = Ytrain[0]["output_1"]

categorical_columns = [
    "input_2", "input_3", "input_4", "input_5",
    "input_6", "input_7", "input_8", "input_9",
    "input_10", "input_11"
]

X_train = Xtrain[0].copy().drop(columns=["input_1"])
X_test  = Xtest[0].copy().drop(columns=["input_1"])

for col in categorical_columns:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("category")
        X_test[col] = X_test[col].astype("category")

print(X_train.dtypes)

print("Training XGBoost...")
xgb_model = train_xgboost(X_train, y_train)
print("Training LightGBM...")
lgb_model = train_lightgbm(X_train, y_train)
print("Training CatBoost...")
categorical_columns = [
    "input_2", "input_3", "input_4", "input_5",
    "input_6", "input_7", "input_8", "input_9",
    "input_10", "input_11"
]
cat_model = train_catboost(X_train, y_train, categorical_columns)

input_2     category
input_3     category
input_4     category
input_5     category
input_6     category
input_7     category
input_8     category
input_9     category
input_10    category
input_11    category
input_14     float64
input_18     float64
input_19     float64
input_20     float64
input_21     float64
input_22     float64
dtype: object
Training XGBoost...
[19:54:31] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (2589, 16, 41424).
Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000445 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1106
[LightGBM] [Info] Number of data points in the train set: 2589, number of used features: 6
[LightGBM] [Info] Start training from score 0.416377
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def test_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("Test results")
    print("------------")
    print(f"MSE  : {mse:.6f}")
    print(f"RMSE : {rmse:.6f}")
    print(f"MAE  : {mae:.6f}")
    print(f"R²   : {r2:.6f}")

    return y_pred
y_test = Ytest[0]["output_1"]
y_pred1 = test_model(xgb_model, X_test, y_test)
y_pred2 = test_model(lgb_model, X_test, y_test)
y_pred3 = test_model(cat_model, X_test, y_test)


Test results
------------
MSE  : 0.090381
RMSE : 0.300634
MAE  : 0.199391
R²   : 0.634208
Test results
------------
MSE  : 0.085176
RMSE : 0.291850
MAE  : 0.192399
R²   : 0.655272
Test results
------------
MSE  : 0.081750
RMSE : 0.285919
MAE  : 0.193470
R²   : 0.669140


# TabPFN. Current SOTA for Tabular Data

In [1]:
!pip install tabpfn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 13.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.9/471.9 kB 20.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 17.4 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 20.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 23.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 20.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 15.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 21.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 18.3 MB/s e

In [ ]:
from tabpfn import TabPFNClassifier, TabPFNRegressor

clf = TabPFNClassifier()
clf.fit(X_train, y_train)  # downloads checkpoint on first use
predictions = clf.predict(X_test)

reg = TabPFNRegressor()
reg.fit(X_train, y_train)  # downloads checkpoint on first use
predictions = reg.predict(X_test)